# Sesión 04 - Distribuciones, PMF y PDF

Objetivo: construir distribuciones discretas y continuas, verificar normalización y calcular probabilidades.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate, stats

rng = np.random.default_rng(11)
pd.set_option("display.precision", 4)


## 1. PMF construida desde un caso de demanda

Supongamos una demanda diaria de 0 a 8 unidades.


### Lectura matemática

- **Distribución asumida:** PMF discreta sobre un soporte finito de demanda.
- **Parámetros estimados/usados:** probabilidades $p_x=P(X=x)$.
- **Supuesto que puede fallar:** soporte truncado o probabilidades no estacionarias.
- **Diagnóstico:** normalización, esperanza, varianza y comparación con frecuencias simuladas.


In [ ]:
demanda = np.arange(0, 9)
probabilidades = np.array([0.03, 0.06, 0.10, 0.16, 0.22, 0.18, 0.13, 0.08, 0.04])
probabilidades = probabilidades / probabilidades.sum()

pmf = pd.DataFrame({"demanda": demanda, "probabilidad": probabilidades})
pmf["cdf"] = pmf["probabilidad"].cumsum()

assert np.isclose(pmf["probabilidad"].sum(), 1)
pmf


In [ ]:
esperanza = np.sum(demanda * probabilidades)
varianza = np.sum((demanda - esperanza) ** 2 * probabilidades)
p_al_menos_5 = probabilidades[demanda >= 5].sum()

print(f"E[X] = {esperanza:.3f}")
print(f"Var(X) = {varianza:.3f}")
print(f"P(X >= 5) = {p_al_menos_5:.3f}")


In [ ]:
muestra_demanda = rng.choice(demanda, size=2_000, p=probabilidades)
frecuencias = pd.Series(muestra_demanda).value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(demanda - 0.15, probabilidades, width=0.3, label="PMF teórica")
ax.bar(frecuencias.index + 0.15, frecuencias.values, width=0.3, label="frecuencia simulada")
ax.set_title("PMF teórica vs simulación")
ax.set_xlabel("demanda")
ax.set_ylabel("probabilidad")
ax.legend()
plt.show()


## 2. PDF y área bajo la curva

Definimos una densidad triangular en $[0, 10]$. La densidad en un punto no es una probabilidad; el área sí.


### Lectura matemática

- **Distribución asumida:** densidad continua triangular en un intervalo acotado.
- **Parámetro estimado:** ninguno; la forma se define analíticamente.
- **Supuesto que puede fallar:** usar densidad como probabilidad puntual.
- **Diagnóstico:** integrar la PDF y comprobar que el área total es 1.


In [ ]:
def densidad_triangular(x):
    x = np.asarray(x)
    y = np.zeros_like(x, dtype=float)
    izquierda = (0 <= x) & (x <= 5)
    derecha = (5 < x) & (x <= 10)
    y[izquierda] = x[izquierda] / 25
    y[derecha] = (10 - x[derecha]) / 25
    return y

area_total, _ = integrate.quad(lambda t: float(densidad_triangular(t)), 0, 10)
p_2_6, _ = integrate.quad(lambda t: float(densidad_triangular(t)), 2, 6)

print(f"Área total = {area_total:.4f}")
print(f"P(2 <= X <= 6) = {p_2_6:.4f}")


In [ ]:
xs = np.linspace(-1, 11, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, densidad_triangular(xs), label="PDF")
zona = (xs >= 2) & (xs <= 6)
ax.fill_between(xs[zona], densidad_triangular(xs[zona]), alpha=0.35, label="P(2 <= X <= 6)")
ax.set_title("Probabilidad como área bajo la PDF")
ax.set_xlabel("x")
ax.set_ylabel("densidad")
ax.legend()
plt.show()


## 3. CDF empírica vs CDF teórica


In [ ]:
normal = stats.norm(loc=100, scale=15)
muestra = normal.rvs(size=1_000, random_state=rng)
xs = np.linspace(50, 150, 300)

muestra_ordenada = np.sort(muestra)
ecdf = np.arange(1, len(muestra_ordenada) + 1) / len(muestra_ordenada)

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(muestra_ordenada, ecdf, where="post", label="CDF empírica")
ax.plot(xs, normal.cdf(xs), color="crimson", label="CDF teórica normal")
ax.set_title("CDF empírica vs teórica")
ax.set_xlabel("x")
ax.set_ylabel("F(x)")
ax.legend()
plt.show()


## 4. Distribuciones empíricas con demanda real

Este bloque reutiliza `sales_data.csv`: demanda como variable discreta amplia y precio como variable continua positiva.


### Lectura matemática

- **Distribución asumida:** no paramétrica empírica para `Demand` y `Price`.
- **Parámetro estimado:** CDF empírica, cuantiles y momentos muestrales.
- **Supuesto que puede fallar:** muestra no representativa o mezcla de segmentos heterogéneos.
- **Diagnóstico:** comparar histogramas por promoción, categoría o clima.


In [ ]:
from pathlib import Path

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró data_sources/sales_data.csv. Se mantiene la sección sintética.")
else:
    ventas_df = pd.read_csv(DATA_DIR / "sales_data.csv", usecols=["Demand", "Price", "Category", "Promotion"])
    pmf_demanda = ventas_df["Demand"].value_counts(normalize=True).sort_index()
    print(f"Filas: {len(ventas_df):,}")
    print(f"Soporte observado de Demand: {ventas_df['Demand'].min()} a {ventas_df['Demand'].max()}")
    print(f"P(Demand >= percentil 90) = {(ventas_df['Demand'] >= ventas_df['Demand'].quantile(0.90)).mean():.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ventas_df["Demand"].hist(bins=40, density=True, ax=axes[0])
    axes[0].set_title("Distribución empírica de Demand")
    ventas_df["Price"].hist(bins=40, density=True, ax=axes[1])
    axes[1].set_title("Distribución empírica de Price")
    plt.show()


## Práctica

Modifica la PMF de demanda para representar un escenario de mayor demanda. Verifica normalización y compara $E[X]$, $Var(X)$ y $P(X \ge 5)$.
